# Notebook 2: EEG data: shapes, masks, and signals solutions


The UCI **EEG Eye State** dataset contains 14 EEG channels from one continuous
117-second measurement, with eye state labelled from video (`0` open, `1` closed).

It contains one participant and is not an `epochs × channels × time` dataset.

Roesler, O. (2013), UCI Machine Learning Repository, CC BY 4.0,
<https://doi.org/10.24432/C57G7J>.

In [ ]:
from pathlib import Path
import sys

# Support running from either the repository root or this notebook folder.
for _candidate in (Path.cwd(), Path.cwd() / "book" / "notebooks"):
    if (_candidate / "workshop_checks.py").exists():
        sys.path.insert(0, str(_candidate))
        break

from workshop_checks import Check, run_checks

check = Check()


## 1: Load the ARFF file

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import arff

candidates = [
    Path("book/data/real/eeg_eye_state.arff"),
    Path("../data/real/eeg_eye_state.arff"),
]
data_path = next((path for path in candidates if path.exists()), None)
if data_path is None:
    raise FileNotFoundError("Open this notebook from the workshop repository.")

raw_records, metadata = arff.loadarff(data_path)
eeg = pd.DataFrame(raw_records)
eeg["eyeDetection"] = eeg["eyeDetection"].astype(int)
eeg.head()

In [ ]:
run_checks("02_eeg_arrays_cell_6", locals())


## 2: Predict the shape

The dataset has 14 EEG channels and one label column. What is the shape of the complete
DataFrame?

- A: `(14980, 14)`
- B: `(14980, 15)`
- C: `(15, 14980)`

In [ ]:
answer_shape = "B"

In [ ]:
run_checks("02_eeg_arrays_cell_11", locals())


## 3: Separate features and target

Fill the blanks so `X` contains the channels and `y` contains eye state.

In [ ]:
X = eeg.drop(columns=["eyeDetection"])
y = eeg["eyeDetection"]

In [ ]:
run_checks("02_eeg_arrays_cell_16", locals())


## 4: Move from pandas to NumPy

Create `signals` as a NumPy array. Then select the first 100 samples from channel O1.
The output should be one-dimensional.

In [ ]:
signals = X.to_numpy()
o1_index = list(X.columns).index("O1")
o1_excerpt = signals[:100, o1_index]

In [ ]:
run_checks("02_eeg_arrays_cell_24", locals())


## 5: Boolean masks

Use `y` to make two arrays: samples recorded with eyes open and samples recorded with
eyes closed.

In [ ]:
eyes_open = signals[y.eq(0)]
eyes_closed = signals[y.eq(1)]

In [ ]:
run_checks("02_eeg_arrays_cell_29", locals())


## 6: Aggregate along the correct axis

Calculate one mean value per channel for each eye state. The result should have shape
`(14,)`.

In [ ]:
open_channel_means = eyes_open.mean(axis=0)
closed_channel_means = eyes_closed.mean(axis=0)

In [ ]:
run_checks("02_eeg_arrays_cell_34", locals())


## 7: Visual comparison

Make a grouped or paired plot comparing the 14 channel means. Label the axes and states.
Then answer: why would this plot alone be insufficient evidence that closing the eyes
*caused* the observed differences?

In [ ]:
# Create a grouped bar chart with one pair of bars per channel.


# Why this does not establish causation:

## Bonus: Build pseudo-epochs

Take the first 14,000 samples and reshape them into
`100 pseudo-epochs × 140 time samples × 14 channels`, then transpose to the ACN
convention `epochs × channels × time`.

These fixed-width chunks are not experimentally defined epochs.

In [ ]:
pseudo_epochs = signals[:14000].reshape(100, 140, 14).transpose(0, 2, 1)

In [ ]:
run_checks("02_eeg_arrays_cell_46", locals())
